# Lesson 03 - 智能代理設計模式

在本課中，我們將探討三個構建高效 AI 代理的基礎設計模式：

1. **清晰的代理指令** — 編寫精確的、定義角色的提示，以指導代理行為
2. **使用 Pydantic 模型的結構化輸出** — 確保代理返回可預測且經過驗證的數據
3. **單一職責代理** — 設計專注於單一任務且表現出色的代理

我們將在**旅遊目的地推薦**場景中應用每個模式，逐步構建一個能建議目的地、檢查可用性及處理後勤的系統。


## 設置


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Pattern 1: 清晰的代理指示

最有影響力的模式也是最簡單的：為你的代理撰寫清晰、詳細的指示。

良好的指示定義了：
- **誰** 是代理（角色和語氣）
- **做什麼**（逐步職責）
- **如何** 表現（限制和風格）

以下，我們創建一個旅遊禮賓代理，並用明確的指示塑造它產生的每個回應。


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: 使用 Pydantic 模型的結構化輸出

自由格式文本適用於對話，但下游系統需要結構化數據。  
通過將 **Pydantic 模型** 與 **工具函數** 配對，我們可以：

- 定義代理輸出的精確架構
- 自動驗證回應
- 可靠地將代理結果整合到應用邏輯中

我們還引入了一個返回目的地詳情的工具，讓代理的推薦基於真實數據。


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: 單一職責代理人

複雜任務受惠於將工作拆分為多個專注的代理人，每個代理人有單一職責：

- 一個 **目的地專家**，了解地點和可用性
- 一個 **後勤規劃員**，負責航班、酒店和行程安排

這與軟件工程中的*關注點分離*原則相呼應 — 每個代理人都更容易獨立測試、維護和改進。


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Summary

In this lesson we applied three agentic design patterns to a travel recommender scenario:

| Pattern | Key Idea | Benefit |
|---|---|---|
| **Clear Instructions** | 預先定義角色、職責和限制 | 一致、符合品牌的代理行為 |
| **Structured Output** | 使用 Pydantic 模型作為回應格式 | 經驗證、機器可讀的結果 |
| **Single Responsibility** | 給每個代理一個專注的工作 | 更易於測試、維護和組合 |

這些模式可以自然組合——你可以將清晰指示與結構化輸出結合於單一職責代理中，以建立健全、適合生產的系統。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件係透過人工智能翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們致力確保準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原文件以其本地語言版本為權威來源。對於重要資訊，建議採用專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
